### OCR client (Run inference server on Kaggle first)

In [2]:
import requests
import json
import sys
import os
import glob
from PIL import Image, ImageDraw, ImageFont, ImageStat
from pdf2image import convert_from_path

# ==========================================
# 1. CONFIGURATION
# ==========================================
# REPLACE THIS WITH YOUR ACTUAL NGROK BASE URL (WITHOUT /predict)
BASE_URL = "https://c438-34-73-131-134.ngrok-free.app"
API_URL = f"{BASE_URL.rstrip('/')}/predict"

OUTPUT_DIR = "output_results"
POPPLER_PATH = r"C:\poppler-25.12.0\Library\bin"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Extra pixels to expand each OCR box on all sides for white-fill output
WHITE_FILL_PADDING = 8

# ==========================================
# 2. DRAWING FUNCTION
# ==========================================
def draw_boxes(image, ocr_blocks):
    """
    Draw OCR bounding boxes with index labels on an image.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        ocr_blocks (list): The list of dictionaries containing OCR bounding boxes.
        
    Returns:
        PIL.Image.Image: The image with drawn OCR boxes.
    """
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except IOError:
        font = ImageFont.load_default()

    for i, block in enumerate(ocr_blocks):
        box = block.get('bbox_2d') 
        text = block.get('text', '')

        if box and len(box) == 4:
            # ✅ Force safe coordinate order so PIL never silently skips drawing
            x0, y0, x1, y1 = box
            safe_box = [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)]
            
            draw.rectangle(safe_box, outline="red", width=3)
            
            text_pos = (safe_box[0], max(0, safe_box[1] - 25))
            display_text = f"{i+1}"
            
            bbox = draw.textbbox(text_pos, display_text, font=font)
            draw.rectangle(bbox, fill="red")
            draw.text(text_pos, display_text, fill="white", font=font)
            
            print(f"  [{i+1}] Found: {text[:30]}... at {box}")

    return image


def fill_boxes_with_white(image, ocr_blocks, padding=0):
    """
    Returns a copy of the image where each OCR bounding box area is filled with white.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        ocr_blocks (list): The list of dictionaries containing OCR bounding boxes.
        padding (int): Pixels to expand each box on all sides.
        
    Returns:
        PIL.Image.Image: The image with white-filled regions.
    """
    white_filled = image.copy()
    draw = ImageDraw.Draw(white_filled)
    width, height = white_filled.size

    for block in ocr_blocks:
        box = block.get('bbox_2d')
        if box and len(box) == 4:
            x0, y0, x1, y1 = [int(v) for v in box]

            x0 = max(0, x0 - padding)
            y0 = max(0, y0 - padding)
            x1 = min(width - 1, x1 + padding)
            y1 = min(height - 1, y1 + padding)

            if x1 > x0 and y1 > y0:
                draw.rectangle([x0, y0, x1, y1], fill="white")

    return white_filled


# ==========================================
# 3. MAIN LOGIC
# ==========================================
def process_file(file_path):
    """
    Process a single file (image or PDF) by sending to OCR API and drawing bounding boxes.
    
    Args:
        file_path (str): The file path to the image or PDF.
        
    Returns:
        None
    """
    print(f"Sending {file_path} to API...")
   
    # 1. Prepare the file for upload
    lower_path = file_path.lower()
    if lower_path.endswith(".pdf"):
        mime_type = "application/pdf"
    elif lower_path.endswith(".png"):
        mime_type = "image/png"
    else:
        mime_type = "image/jpeg"
   
    # 2. Make the POST request
    try:
        with open(file_path, 'rb') as file_handle: # read as binary
            files = {
                'file': (os.path.basename(file_path), file_handle, mime_type) # maps with file: UploadFile = File(...)
            }
            # data = {"prompt": "optional instruction"}  # optional maps with prompt: Optional[str] = Form(None)
            # response = requests.post(API_URL, files=files, data=data, timeout=600)
            response = requests.post(API_URL, files=files, timeout=600) # High timeout for large PDFs
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] API Error: {e}")
        return
    except OSError as e:
        print(f"[ERROR] File Error: {e}")
        return

    if data.get("status") != "success":
        print(f"[ERROR] Server Error: {data}")
        return

    results = data["results"]
    print(f"Received data for {len(results)} page(s). Processing visualization...")

    # 3. Handle PDF vs Image Visualization
    if file_path.lower().endswith(".pdf"):
        # Convert local PDF to images so we can draw on them
        print("   Converting local PDF to images for drawing...")
        pdf_images = convert_from_path(file_path, poppler_path=POPPLER_PATH)
       
        for res in results:
            page_num = res['page']
            ocr_data = res['data']['ocr_blocks']
           
            # Get the corresponding image (Page 1 is index 0)
            if page_num <= len(pdf_images):
                img = pdf_images[page_num - 1].convert("RGB")

                # Save annotated image (red boxes + IDs)
                img_with_boxes = draw_boxes(img.copy(), ocr_data)
                annotated_filename = f"{os.path.basename(file_path)}_page_{page_num}_analyzed.jpg"
                annotated_path = os.path.join(OUTPUT_DIR, annotated_filename)
                img_with_boxes.save(annotated_path)
                print(f"   Saved: {annotated_path}")

                # Save white-filled image (text areas removed)
                white_filled_img = fill_boxes_with_white(img.copy(), ocr_data, padding=WHITE_FILL_PADDING)
                white_filename = f"{os.path.basename(file_path)}_page_{page_num}_whitefill.jpg"
                white_path = os.path.join(OUTPUT_DIR, white_filename)
                white_filled_img.save(white_path)
                print(f"   Saved: {white_path}")

    else:
        # Handle Single Image
        res = results[0] # There is only 1 result for an image
        ocr_data = res['data']['ocr_blocks']
       
        with Image.open(file_path) as img:
            img = img.convert("RGB")

            # Save annotated image (red boxes + IDs)
            img_with_boxes = draw_boxes(img.copy(), ocr_data)
            annotated_filename = f"{os.path.basename(file_path)}_analyzed.jpg"
            annotated_path = os.path.join(OUTPUT_DIR, annotated_filename)
            img_with_boxes.save(annotated_path)
            print(f"   Saved: {annotated_path}")

            # Save white-filled image (text areas removed)
            white_filled_img = fill_boxes_with_white(img.copy(), ocr_data, padding=WHITE_FILL_PADDING)
            white_filename = f"{os.path.basename(file_path)}_whitefill.jpg"
            white_path = os.path.join(OUTPUT_DIR, white_filename)
            white_filled_img.save(white_path)
            print(f"   Saved: {white_path}")


def collect_input_files(inputs):
    """
    Collect valid image/PDF files from exact paths, folders, or glob patterns.
    
    Args:
        inputs (list): The list of input strings.
        
    Returns:
        list: The list of collected unique file paths.
    """
    allowed_ext = {".png", ".jpg", ".jpeg", ".pdf"}
    collected = []

    for item in inputs:

        # INPUTS = ["images"]
        if os.path.isdir(item):
            for name in sorted(os.listdir(item)):
                path = os.path.join(item, name)
                ext = os.path.splitext(path)[1].lower()
                if os.path.isfile(path) and ext in allowed_ext:
                    collected.append(path)

        # INPUTS = ["images\\*.png"]
        elif any(ch in item for ch in ["*", "?", "["]):
            for path in sorted(glob.glob(item)):
                ext = os.path.splitext(path)[1].lower()
                if os.path.isfile(path) and ext in allowed_ext:
                    collected.append(path)

        # INPUTS = ["images\\85.png", "images\\86.jpg"]
        elif os.path.isfile(item):
            ext = os.path.splitext(item)[1].lower()
            if ext in allowed_ext:
                collected.append(item)
            else:
                print(f"[WARN] Skipping unsupported file type: {item}")
        else:
            print(f"[WARN] Not found: {item}")

    # Remove duplicates while preserving order
    return list(dict.fromkeys(collected))


# ==========================================
# 4. RUNNER
# ==========================================
if __name__ == "__main__":
    # Examples:
    # INPUTS = ["images\\85.png", "images\\86.jpg"]
    # INPUTS = ["images"]  # process all supported files in folder
    # INPUTS = ["images\\*.png"]
    INPUTS = ["images\\0.jpg"]

    files_to_process = collect_input_files(INPUTS)

    if not files_to_process:
        print("[ERROR] No valid input files found.")
    else:
        print(f"Found {len(files_to_process)} file(s) to process.")
        for idx, file_path in enumerate(files_to_process, start=1):
            print(f"\n[{idx}/{len(files_to_process)}] Processing: {file_path}")
            process_file(file_path)

Found 1 file(s) to process.

[1/1] Processing: images\0.jpg
Sending images\0.jpg to API...
Received data for 1 page(s). Processing visualization...
  [1] Found: あっこの前校内に出て騒ぎになったやつだ... at [897, 719, 1001, 934]
  [2] Found: スロウ様とはぜんぜん似て・・・うーん・・・... at [681, 815, 781, 984]
  [3] Found: はっきり否定してくれ頼むから... at [189, 729, 291, 871]
  [4] Found: うわっ・・・こいつは色が気持ち悪い・・・... at [938, 1119, 1040, 1261]
  [5] Found: え?可愛いと思いますけど・・・... at [755, 1111, 854, 1300]
  [6] Found: ・・・何をしていますのあの子たち... at [517, 1325, 575, 1511]
  [7] Found: うーん... at [154, 1384, 175, 1434]
  [8] Found: 豚公爵に転生したから、今度は君に好きと言いたい... at [127, 1518, 430, 1585]
  [9] Found: わー・・・色々載ってますね... at [621, 284, 754, 497]
  [10] Found: 見たことのないモンスターも多いですね... at [185, 400, 287, 565]
   Saved: output_results\0.jpg_analyzed.jpg
   Saved: output_results\0.jpg_whitefill.jpg


### Translation client (Run inference server on Kaggle first)

In [ ]:
import requests
from tqdm.auto import tqdm

def translate_sentences(sentences: list, api_url: str = "https://e80e-34-23-28-216.ngrok-free.app/translate"):
    """
    Send a list of Japanese sentences to the translation API and return the results.
    
    Args:
        sentences (list): The list of Japanese sentence strings to translate.
        api_url (str): The translation endpoint URL.
        
    Returns:
        list: The list of translated Thai strings.
    """
    translations = []
    
    # 1. Define the headers to bypass ngrok's free tier warning
    headers = {
        "ngrok-skip-browser-warning": "true"
    }
    
    for sentence in tqdm(sentences, desc="Translating Sentences"):
        payload = {"japanese_text": sentence}
        
        try:
            # 2. Pass the headers into the POST request
            response = requests.post(api_url, json=payload, headers=headers, timeout=120)
            response.raise_for_status() 
            
            result = response.json()
            translations.append(result["translated_text"])
            
        except requests.exceptions.RequestException as e:
            print(f"\n[Error] Failed to translate: '{sentence[:20]}...' -> {e}")
            translations.append("[ERROR: Translation Failed]")
            
    return translations

In [ ]:
# 1. Define your list of Japanese sentences
my_manga_texts = [
    "おはようございます！今日も頑張りましょう！",
    "まさか…あいつが犯人だったなんて…",
    "絶対に諦めない！"
]

# 2. Call the function
thai_translations = translate_sentences(my_manga_texts)

# 3. Print the results
for jp, th in zip(my_manga_texts, thai_translations):
    print(f"JP: {jp}")
    print(f"TH: {th}\n")

### Full-client

In [ ]:
import os
import json
import glob
import io
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm
from PIL import Image, ImageDraw, ImageFont, ImageStat
from pdf2image import convert_from_path

# Set different endpoints here
OCR_API_URL = "https://your-ocr-endpoint.ngrok-free.app/predict"
TRANSLATION_API_URL = "https://your-translation-endpoint.ngrok-free.app/translate"
OUTPUT_DIR = globals().get("OUTPUT_DIR", "output_results")
POPPLER_PATH = r"C:\poppler-25.12.0\Library\bin"

# Rendering settings
FONT_PATH = "C:\\Users\\pakor\\AppData\\Local\\Microsoft\\Windows\\Fonts\\LayijiMahaniyomV1.ttf"
TEXT_PADDING = 8
TONE_MARK_VERTICAL_SCALE = 0.65
LINE_SPACING_PX = 4
TOP_MARGIN_PX = 20
RIGHT_PADDING_EXTRA_PX = 6
TOP_PADDING_RATIO = 0.5

# OCR networking defaults
DEFAULT_OCR_CONNECT_TIMEOUT_SEC = 20
DEFAULT_OCR_READ_TIMEOUT_SEC = 300
DEFAULT_OCR_RETRIES = 2
DEFAULT_OCR_JPEG_QUALITY = 95
ALLOWED_EXT = {".png", ".jpg", ".jpeg", ".pdf"}


def debug_log(stage, message):
    """
    Print a structured debug message tagged by pipeline stage.
    
    Args:
        stage (str): The pipeline stage (e.g., 'OCR', 'TRANSLATE').
        message (str): The debug message to print.
        
    Returns:
        None
    """
    print(f"[DEBUG][{stage}] {message}")


def collect_input_files(inputs):
    """
    Collect valid image/PDF files from explicit paths, folders, and glob patterns.
    
    Args:
        inputs (list): The list of input strings containing paths, folder names, or glob patterns.
        
    Returns:
        list: The list of unique, valid file paths collected.
    """
    debug_log("INPUT", f"Collecting files from {len(inputs)} input item(s)")
    collected = []

    for item in inputs:
        item = item.strip()
        if not item:
            continue

        if os.path.isdir(item):
            for name in sorted(os.listdir(item)):
                path = os.path.join(item, name)
                ext = os.path.splitext(path)[1].lower()
                if os.path.isfile(path) and ext in ALLOWED_EXT:
                    collected.append(path)
        elif any(ch in item for ch in ["*", "?", "["]):
            for path in sorted(glob.glob(item)):
                ext = os.path.splitext(path)[1].lower()
                if os.path.isfile(path) and ext in ALLOWED_EXT:
                    collected.append(path)
        elif os.path.isfile(item):
            ext = os.path.splitext(item)[1].lower()
            if ext in ALLOWED_EXT:
                collected.append(item)
            else:
                print(f"[WARN] Skipping unsupported file type: {item}")
        else:
            print(f"[WARN] Not found: {item}")

    return list(dict.fromkeys(collected))


def run_ocr(file_path: str, ocr_api_url: str):
    """
    Upload a file path directly to the OCR API and return parsed JSON.
    
    Args:
        file_path (str): The file path to the image or PDF.
        ocr_api_url (str): The endpoint URL for the OCR service.
        
    Returns:
        dict: The parsed JSON data from the OCR API.
    """
    debug_log("OCR", f"Uploading file path: {file_path}")
    lower_path = file_path.lower()
    if lower_path.endswith(".pdf"):
        mime_type = "application/pdf"
    elif lower_path.endswith(".png"):
        mime_type = "image/png"
    else:
        mime_type = "image/jpeg"

    headers = {"ngrok-skip-browser-warning": "true"}

    with open(file_path, "rb") as file_handle:
        files = {
            "file": (os.path.basename(file_path), file_handle, mime_type)
        }
        response = requests.post(ocr_api_url, files=files, headers=headers, timeout=300)

    response.raise_for_status()
    data = response.json()

    if data.get("status") != "success":
        raise ValueError(f"OCR server returned non-success response: {data}")

    return data


def run_ocr_for_pil_image(
    image,
    upload_name: str,
    ocr_api_url: str,
    connect_timeout: int = DEFAULT_OCR_CONNECT_TIMEOUT_SEC,
    read_timeout: int = DEFAULT_OCR_READ_TIMEOUT_SEC,
    max_retries: int = DEFAULT_OCR_RETRIES,
    jpeg_quality: int = DEFAULT_OCR_JPEG_QUALITY,
):
    """
    Upload a PIL image to OCR API with retry/backoff and timeout controls.
    
    Args:
        image (PIL.Image.Image): The PIL image to upload.
        upload_name (str): The name to assign to the uploaded file.
        ocr_api_url (str): The endpoint URL for the OCR service.
        connect_timeout (int): Timeout in seconds for connection.
        read_timeout (int): Timeout in seconds for reading the response.
        max_retries (int): The maximum number of retry attempts.
        jpeg_quality (int): The JPEG compression quality.
        
    Returns:
        dict: The parsed JSON data from the OCR API.
    """
    debug_log("OCR", f"Uploading in-memory image: {upload_name}")
    headers = {"ngrok-skip-browser-warning": "true", "Connection": "close"}

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=int(jpeg_quality))
    # Cache bytes once so each retry can rebuild a fresh readable stream.
    files_payload = buffer.getvalue()

    last_error = None
    total_attempts = max(1, int(max_retries)) + 1

    for attempt in range(total_attempts):
        file_buffer = io.BytesIO(files_payload)
        files = {"file": (f"{upload_name}.jpg", file_buffer, "image/jpeg")}
        debug_log(
            "OCR",
            (
                f"Attempt {attempt + 1}/{total_attempts} -> {ocr_api_url} "
                f"(connect={connect_timeout}s, read={read_timeout}s)"
            ),
        )

        try:
            response = requests.post(
                ocr_api_url,
                files=files,
                headers=headers,
                timeout=(connect_timeout, read_timeout),
            )
            response.raise_for_status()
            data = response.json()
            if data.get("status") != "success":
                raise ValueError(f"OCR server returned non-success response: {data}")
            return data
        except requests.exceptions.Timeout as exc:
            last_error = exc
            debug_log("ERROR", f"OCR timeout for {upload_name}: {exc}")
        except requests.exceptions.RequestException as exc:
            last_error = exc
            debug_log("ERROR", f"OCR request failed for {upload_name}: {exc}")

        if attempt < total_attempts - 1:
            backoff = min(5, attempt + 1)
            debug_log("OCR", f"Retrying after {backoff}s")
            time.sleep(backoff)
    else:
        raise TimeoutError(
            "OCR request failed after retries. Check OCR API health/network or increase timeout."
        ) from last_error


def run_translation(sentences: list, translation_api_url: str):
    """
    Translate Japanese sentences one-by-one via the translation API.
    
    Args:
        sentences (list): The list of Japanese sentence strings.
        translation_api_url (str): The endpoint URL for the translation service.
        
    Returns:
        list: The list of translated text strings.
    """
    debug_log("TRANSLATE", f"Translating {len(sentences)} sentence(s)")
    headers = {"ngrok-skip-browser-warning": "true"}
    translations = []

    for idx, sentence in enumerate(sentences, start=1):
        payload = {"japanese_text": sentence}
        debug_log("TRANSLATE", f"Request {idx}/{len(sentences)}")
        response = requests.post(translation_api_url, json=payload, headers=headers, timeout=120)
        response.raise_for_status()
        result = response.json()
        translations.append(result.get("translated_text", ""))

    return translations


def load_font(font_path: str, size: int):
    """
    Load a TrueType font and fallback to PIL default if unavailable.
    
    Args:
        font_path (str): The file path to the TrueType font.
        size (int): The font size to load.
        
    Returns:
        PIL.ImageFont: The loaded font object.
    """
    try:
        return ImageFont.truetype(font_path, size)
    except OSError:
        print(f"[WARN] Font not found at '{font_path}'. Using default font.")
        return ImageFont.load_default()


def text_size(draw, text, font):
    """
    Measure rendered text width and height for the given font.
    
    Args:
        draw (PIL.ImageDraw.Draw): The PIL ImageDraw object.
        text (str): The text string to measure.
        font (PIL.ImageFont): The font object to use.
        
    Returns:
        tuple: The width and height of the measured text.
    """
    box = draw.textbbox((0, 0), text, font=font)
    return box[2] - box[0], box[3] - box[1]


def compute_line_metrics(draw, font, line_spacing=0, tone_mark_scale=1.0):
    """
    Compute rendered line height and per-line step after scaling/spacing.
    
    Args:
        draw (PIL.ImageDraw.Draw): The PIL ImageDraw object.
        font (PIL.ImageFont): The font object to use.
        line_spacing (int): The vertical spacing between lines.
        tone_mark_scale (float): The vertical scale adjustment for tone marks.
        
    Returns:
        tuple: The rendered line height and the step size to the next line.
    """
    _, base_line_h = text_size(draw, "Ag", font)
    base_line_h = max(1, base_line_h)
    rendered_line_h = max(1, int(round(base_line_h * max(0.5, tone_mark_scale))))
    line_step = rendered_line_h + max(0, int(line_spacing))
    return rendered_line_h, max(1, line_step)


def wrap_text_for_box(draw, text, font, max_width):
    """
    Wrap text by character so lines fit inside max_width. No height limits here.
    
    Args:
        draw (PIL.ImageDraw.Draw): The PIL ImageDraw object.
        text (str): The text string to wrap.
        font (PIL.ImageFont): The font object to use.
        max_width (int): The maximum allowed width for a single line of text.
        
    Returns:
        list: The list of text strings representing each wrapped line.
    """
    if not text:
        return []

    lines = []
    current = ""

    for ch in text:
        candidate = current + ch
        candidate_width, _ = text_size(draw, candidate, font)
        if candidate_width <= max_width or not current:
            current = candidate
        else:
            lines.append(current)
            current = ch

    if current:
        lines.append(current)

    return lines


def fit_and_wrap_text(
    draw,
    text,
    font_path,
    max_width,
    max_height,
    start_size=60,
    min_size=14,
    line_spacing=0,
    tone_mark_scale=1.0,
):
    """
    Find a fitting font size and wrapped lines for a constrained text box.
    
    Args:
        draw (PIL.ImageDraw.Draw): The PIL ImageDraw object.
        text (str): The text string to fit and wrap.
        font_path (str): The path to the font to use.
        max_width (int): The maximum width of the text box.
        max_height (int): The maximum height of the text box.
        start_size (int): The starting font size for scaling down.
        min_size (int): The minimum allowable font size.
        line_spacing (int): The vertical spacing between lines.
        tone_mark_scale (float): The vertical scale adjustment for tone marks.
        
    Returns:
        tuple: A combination of (font, lines, line_height, line_step).
    """
    size = max(min_size, start_size)
    while size > min_size:
        font = load_font(font_path, size)
        line_h, line_step = compute_line_metrics(
            draw, font, line_spacing=line_spacing, tone_mark_scale=tone_mark_scale
        )
        lines = wrap_text_for_box(draw, text, font, max_width)
        total_h = line_h + (max(1, len(lines)) - 1) * line_step
        too_wide = any(text_size(draw, ln, font)[0] > max_width for ln in lines if ln)

        if lines and (not too_wide) and total_h <= max_height:
            # We found a fitting size! Decrease it by 4 for a safety margin.
            final_size = max(min_size, size - 6)
            if final_size != size:
                font = load_font(font_path, final_size)
                line_h, line_step = compute_line_metrics(
                    draw, font, line_spacing=line_spacing, tone_mark_scale=tone_mark_scale
                )
                lines = wrap_text_for_box(draw, text, font, max_width)
            return font, lines, line_h, line_step
        size -= 1

    # Fallback to min_size
    font = load_font(font_path, min_size)
    line_h, line_step = compute_line_metrics(
        draw, font, line_spacing=line_spacing, tone_mark_scale=tone_mark_scale
    )
    lines = wrap_text_for_box(draw, text, font, max_width)
    return font, lines, line_h, line_step


def draw_scaled_text_line(image, position, text, font, fill="black", y_scale=1.0):
    """
    Draw one line of text with optional vertical scaling to compact glyph marks.
    
    Args:
        image (PIL.Image.Image): The PIL image to draw on.
        position (tuple): The (x, y) coordinates for the top-left of the text.
        text (str): The text to draw.
        font (PIL.ImageFont): The font to use for drawing.
        fill (str, optional): The fill color around the text. Defaults to 'black'.
        y_scale (float, optional): The vertical scaling factor for the text. Defaults to 1.0.
        
    Returns:
        None
    """
    x, y = position
    if y_scale >= 0.99:
        ImageDraw.Draw(image).text((x, y), text, fill=fill, font=font)
        return

    # Draw on transparent layer first, then resize only text pixels (not full image).
    temp = Image.new("RGBA", image.size, (0, 0, 0, 0))
    temp_draw = ImageDraw.Draw(temp)
    temp_draw.text((x, y), text, fill=fill, font=font)
    box = temp_draw.textbbox((x, y), text, font=font)
    if not box:
        return

    crop = temp.crop(box)
    new_h = max(1, int(round(crop.height * max(0.5, y_scale))))
    if new_h != crop.height:
        # Bicubic keeps Thai marks smoother after vertical compression.
        crop = crop.resize((crop.width, new_h), Image.Resampling.BICUBIC)

    paste_x = box[0]
    paste_y = box[3] - new_h
    image.paste(crop, (paste_x, paste_y), crop)


def draw_ocr_boxes(image, ocr_blocks):
    """
    Draw OCR polygons (quad) or bbox fallback with index labels on an image.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        ocr_blocks (list): The list of OCR block dictionaries containing bounding box data.
        
    Returns:
        PIL.Image.Image: The image with drawn OCR boxes.
    """
    draw = ImageDraw.Draw(image)
    for idx, block in enumerate(ocr_blocks, start=1):
        quad = block.get("quad")
        box = block.get("bbox_2d")

        anchor_x = None
        anchor_y = None
        if quad and len(quad) == 8:
            points = [int(v) for v in quad]
            polygon = [
                (points[0], points[1]),
                (points[2], points[3]),
                (points[4], points[5]),
                (points[6], points[7]),
            ]
            draw.polygon(polygon, outline="red", width=4)
            anchor_x, anchor_y = polygon[0]
        elif box and len(box) == 4:
            x0, y0, x1, y1 = [int(v) for v in box]
            draw.rectangle([x0, y0, x1, y1], outline="red", width=3)
            anchor_x, anchor_y = x0, y0
        else:
            continue

        label = str(idx)
        font = load_font(FONT_PATH, 30)
        label_box = draw.textbbox((anchor_x, max(0, anchor_y - 20)), label, font=font)
        draw.rectangle(label_box, fill="red")
        draw.text((anchor_x, max(0, anchor_y - 20)), label, fill="white", font=font)
    return image


def fill_boxes_with_white(image, ocr_blocks, padding=8, right_extra=0):
    """
    Fill OCR regions with white, using asymmetric padding around each box.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        ocr_blocks (list): The list of dictionaries containing OCR bounding boxes.
        padding (int): The base padding for white filling.
        right_extra (int): The additional space to clear on the right side.
        
    Returns:
        PIL.Image.Image: The image with white-filled areas.
    """
    draw = ImageDraw.Draw(image)
    width, height = image.size

    for block in ocr_blocks:
        box = block.get("bbox_2d")
        if not box or len(box) != 4:
            continue

        x0, y0, x1, y1 = [int(v) for v in box]
        # Keep top expansion smaller so we don't erase artwork above speech text.
        # top_padding = max(0, int(round(padding * TOP_PADDING_RATIO)))
        x0 = max(0, x0 - padding)
        y0 = max(0, y0 - padding)
        x1 = min(width - 1, x1 + padding + max(0, int(right_extra)))
        y1 = min(height - 1, y1 + padding)

        if x1 > x0 and y1 > y0:
            draw.rectangle([x0, y0, x1, y1], fill="white")

    return image


def is_dark_box(image, box, threshold=70):
    """
    Return True if the mean luminance in the box is below a darkness threshold.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        box (tuple): A tuple containing coords (x0, y0, x1, y1) defining the box.
        threshold (int): The luminance threshold.
        
    Returns:
        bool: True if box is darker than threshold, False otherwise.
    """
    x0, y0, x1, y1 = box
    if x1 <= x0 or y1 <= y0:
        return False

    crop = image.crop((x0, y0, x1 + 1, y1 + 1)).convert("L")
    # Use average grayscale luminance to decide white-vs-black text rendering.
    mean_luma = ImageStat.Stat(crop).mean[0]
    return mean_luma < threshold


def print_debug_pairs(file_path, page_num, source_texts, translated_texts):
    """
    Print OCR and translation text pairs for per-page debugging.
    
    Args:
        file_path (str): The path to the file.
        page_num (int): The page number.
        source_texts (list): The list of original Japanese texts.
        translated_texts (list): The list of translated texts.
        
    Returns:
        None
    """
    print(f"[DEBUG] File={file_path} Page={page_num} OCR blocks={len(source_texts)}")
    for idx, (jp, th) in enumerate(zip(source_texts, translated_texts), start=1):
        print(f"[DEBUG]   #{idx} OCR: {jp}")
        print(f"[DEBUG]   #{idx} TH : {th}")


def draw_translated_text_in_boxes(
    image,
    ocr_blocks,
    translated_texts,
    font_path: str,
    padding: int = 8,
    line_spacing: int = LINE_SPACING_PX,
    tone_mark_scale: float = TONE_MARK_VERTICAL_SCALE,
    top_margin: int = TOP_MARGIN_PX,
    right_padding_extra: int = RIGHT_PADDING_EXTRA_PX,
):
    """
    Render translated text into OCR boxes with adaptive colors and layout fitting.
    
    Args:
        image (PIL.Image.Image): The original PIL image.
        ocr_blocks (list): The list of dictionaries containing OCR bounding boxes.
        translated_texts (list): The list of translated text strings.
        font_path (str): The path to the font to use.
        padding (int): The base padding for text blocks.
        line_spacing (int): The vertical spacing between text lines.
        tone_mark_scale (float): The vertical scale adjustment for tone marks.
        top_margin (int): The top margin for the text.
        right_padding_extra (int): The additional right-side padding.
        
    Returns:
        PIL.Image.Image: The image with rendered translations.
    """
    debug_log("RENDER", f"Rendering translated text into {len(ocr_blocks)} box(es)")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    source_for_color = image.copy()

    for block, translated_text in zip(ocr_blocks, translated_texts):
        box = block.get("bbox_2d")
        if not box or len(box) != 4:
            continue

        x0, y0, x1, y1 = [int(v) for v in box]
        # Match whitefill behavior: less top padding, extra right-side room.
        top_padding = max(0, int(round(padding * TOP_PADDING_RATIO)))
        x0 = max(0, x0 - padding)
        y0 = max(0, y0 - top_padding)
        x1 = min(width - 1, x1 + padding + max(0, int(right_padding_extra)))
        y1 = min(height - 1, y1 + padding)

        if x1 <= x0 or y1 <= y0:
            continue

        dark_box = is_dark_box(source_for_color, (x0, y0, x1, y1))
        # Invert text/background in dark regions for readability.
        bg_fill = "black" if dark_box else "white"
        text_fill = "white" if dark_box else "black"
        draw.rectangle([x0, y0, x1, y1], fill=bg_fill)

        box_w = x1 - x0
        box_h = y1 - y0
        # Start with a slightly smaller size than the box dimensions to leave some margin
        start_size = max(14, int(box_h * 0.9))
        font, lines, line_h, line_step = fit_and_wrap_text(
            draw,
            translated_text,
            font_path,
            box_w,
            box_h,
            start_size=start_size,
            line_spacing=line_spacing,
            tone_mark_scale=tone_mark_scale,
        )

        if not lines:
            continue

        # Dividing by 2 is the standard mathematical trick for centering an object within an enclosed area.
        # Let's say your OCR bubble width (box_w) is 100 pixels.
        # The length of the text string width (line_w) is 60 pixels.
        # (100 - 60) = 40 pixels of empty, unused background space (space before the starts (0-39)).
        # If you don't divide by 2 and just draw at x = 40:

        # The text starts at 40 and extends 60 pixels.
        # The text goes from 40 to 100.
        # All 40 pixels of empty space are on the left side, pushing the text completely to the right edge.

        # If you divide by 2 ((100 - 60) // 2 = 20):
        # Your starting x coordinate becomes exactly 20.
        # The text starts at pixel 20 and extends 60 pixels to pixel 80.
        # This leaves exactly 20 pixels of empty space on the left side, and 20 pixels of empty space on the right side (100 - 80).
        # The text is now perfectly centered! 60 pixels of text are perfectly in the middle.

        # Calculate total height of the text block to center vertically
        total_text_h = line_h + (len(lines) - 1) * line_step
        start_y = y0 + max(0, (box_h - total_text_h) // 2)

        y = start_y
        for line in lines:
            if y + line_h > y1 and y != start_y:
                break
            
            # Calculate width of the current line to center horizontally
            line_w, _ = text_size(draw, line, font)
            line_x = x0 + max(0, (box_w - line_w) // 2)

            draw_scaled_text_line(
                image,
                (line_x, y),
                line,
                font=font,
                fill=text_fill,
                y_scale=tone_mark_scale,
            )
            y += line_step

    return image


def process_single_file(
    file_path: str,
    ocr_api_url: str,
    translation_api_url: str,
    output_dir: str,
    font_path: str = FONT_PATH,
    padding: int = TEXT_PADDING,
    ocr_connect_timeout: int = DEFAULT_OCR_CONNECT_TIMEOUT_SEC,
    ocr_read_timeout: int = DEFAULT_OCR_READ_TIMEOUT_SEC,
    ocr_retries: int = DEFAULT_OCR_RETRIES,
    ocr_jpeg_quality: int = DEFAULT_OCR_JPEG_QUALITY,
):
    """
    Run OCR, translation, rendering, and artifact export for one image/PDF file.
    
    Args:
        file_path (str): The path to the input file.
        ocr_api_url (str): The URL for the OCR API.
        translation_api_url (str): The URL for the translation API.
        output_dir (str): The directory to save processed outputs.
        font_path (str): The path to the font used for rendering.
        padding (int): The base padding for text blocks.
        ocr_connect_timeout (int): The timeout in seconds for OCR API connection.
        ocr_read_timeout (int): The timeout in seconds for reading OCR API response.
        ocr_retries (int): Number of retries for OCR requests.
        ocr_jpeg_quality (int): JPEG quality setting for image compression before OCR.
        
    Returns:
        list: A list of dict pairs mapping 'jp' source text to 'th' translated text and meta info.
    """
    debug_log("PIPELINE", f"Start processing: {file_path}")
    all_pairs = []

    is_pdf = file_path.lower().endswith(".pdf")
    pdf_images = convert_from_path(file_path, poppler_path=POPPLER_PATH) if is_pdf else None

    if is_pdf:
        for page_num, page_image in enumerate(pdf_images, start=1):
            debug_log("PDF", f"Processing page {page_num}/{len(pdf_images)}")
            page_img = page_image.convert("RGB")
            ocr_response = run_ocr_for_pil_image(
                page_img,
                upload_name=f"{Path(file_path).stem}_page_{page_num}",
                ocr_api_url=ocr_api_url,
                connect_timeout=ocr_connect_timeout,
                read_timeout=ocr_read_timeout,
                max_retries=ocr_retries,
                jpeg_quality=ocr_jpeg_quality,
            )
            page_result = (ocr_response.get("results") or [{}])[0]
            blocks = page_result.get("data", {}).get("ocr_blocks", [])
            valid_blocks = [b for b in blocks if (b.get("text") or "").strip()]
            source_texts = [(b.get("text") or "").strip() for b in valid_blocks]
            translated_texts = run_translation(source_texts, translation_api_url) if source_texts else []
            print_debug_pairs(file_path, page_num, source_texts, translated_texts)

            for jp, th in zip(source_texts, translated_texts):
                all_pairs.append({"file": file_path, "page": page_num, "jp": jp, "th": th})

            base_name = Path(file_path).stem

            analyzed = draw_ocr_boxes(page_img.copy(), valid_blocks)
            analyzed_name = f"{base_name}_page_{page_num}_analyzed.jpg"
            analyzed_path = os.path.join(output_dir, analyzed_name)
            analyzed.save(analyzed_path)
            debug_log("SAVE", f"Analyzed image saved: {analyzed_path}")

            whitefilled = fill_boxes_with_white(
                page_img.copy(),
                valid_blocks,
                padding=padding,
                right_extra=RIGHT_PADDING_EXTRA_PX,
            )
            white_name = f"{base_name}_page_{page_num}_whitefill.jpg"
            white_path = os.path.join(output_dir, white_name)
            whitefilled.save(white_path)
            debug_log("SAVE", f"Whitefill image saved: {white_path}")

            rendered = draw_translated_text_in_boxes(
                page_img,
                valid_blocks,
                translated_texts,
                font_path=font_path,
                padding=padding,
            )
            out_name = f"{base_name}_page_{page_num}_translated.jpg"
            out_path = os.path.join(output_dir, out_name)
            rendered.save(out_path)
            debug_log("SAVE", f"Translated image saved: {out_path}")
    else:
        debug_log("IMAGE", f"Processing image file: {file_path}")
        with Image.open(file_path) as source_image:
            image = source_image.convert("RGB")

        ocr_response = run_ocr_for_pil_image(
            image,
            upload_name=Path(file_path).name,
            ocr_api_url=ocr_api_url,
            connect_timeout=ocr_connect_timeout,
            read_timeout=ocr_read_timeout,
            max_retries=ocr_retries,
            jpeg_quality=ocr_jpeg_quality,
        )
        page_result = (ocr_response.get("results") or [{}])[0]
        blocks = page_result.get("data", {}).get("ocr_blocks", [])
        valid_blocks = [b for b in blocks if (b.get("text") or "").strip()]
        source_texts = [(b.get("text") or "").strip() for b in valid_blocks]
        translated_texts = run_translation(source_texts, translation_api_url) if source_texts else []
        print_debug_pairs(file_path, 1, source_texts, translated_texts)

        for jp, th in zip(source_texts, translated_texts):
            all_pairs.append({"file": file_path, "page": 1, "jp": jp, "th": th})

        base_name = Path(file_path).stem

        analyzed = draw_ocr_boxes(image.copy(), valid_blocks)
        analyzed_name = f"{base_name}_analyzed.jpg"
        analyzed_path = os.path.join(output_dir, analyzed_name)
        analyzed.save(analyzed_path)
        debug_log("SAVE", f"Analyzed image saved: {analyzed_path}")

        whitefilled = fill_boxes_with_white(
            image.copy(),
            valid_blocks,
            padding=padding,
            right_extra=RIGHT_PADDING_EXTRA_PX,
        )
        white_name = f"{base_name}_whitefill.jpg"
        white_path = os.path.join(output_dir, white_name)
        whitefilled.save(white_path)
        debug_log("SAVE", f"Whitefill image saved: {white_path}")

        rendered = draw_translated_text_in_boxes(
            image,
            valid_blocks,
            translated_texts,
            font_path=font_path,
            padding=padding,
        )
        out_name = f"{base_name}_translated.jpg"
        out_path = os.path.join(output_dir, out_name)
        rendered.save(out_path)
        debug_log("SAVE", f"Translated image saved: {out_path}")

    json_name = f"{Path(file_path).stem}_jp_th.json"
    json_path = os.path.join(output_dir, json_name)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_pairs, f, ensure_ascii=False, indent=2)
    debug_log("SAVE", f"JSON saved: {json_path}")
    debug_log("PIPELINE", f"Finished processing: {file_path}")

    return all_pairs


def process_multiple_files_with_new_apis(
    inputs,
    ocr_api_url: str = OCR_API_URL,
    translation_api_url: str = TRANSLATION_API_URL,
    output_dir: str = OUTPUT_DIR,
    font_path: str = FONT_PATH,
    padding: int = TEXT_PADDING,
    ocr_connect_timeout: int = DEFAULT_OCR_CONNECT_TIMEOUT_SEC,
    ocr_read_timeout: int = DEFAULT_OCR_READ_TIMEOUT_SEC,
    ocr_retries: int = DEFAULT_OCR_RETRIES,
    ocr_jpeg_quality: int = DEFAULT_OCR_JPEG_QUALITY,
):
    """
    Process multiple files sequentially through the end-to-end translation pipeline.
    
    Args:
        inputs (list): The list of input strings containing paths, folder names, or glob patterns.
        ocr_api_url (str): The URL for the OCR API.
        translation_api_url (str): The URL for the translation API.
        output_dir (str): The directory to save processed outputs.
        font_path (str): The path to the font used for rendering.
        padding (int): The base padding for text blocks.
        ocr_connect_timeout (int): The timeout in seconds for OCR API connection.
        ocr_read_timeout (int): The timeout in seconds for reading OCR API response.
        ocr_retries (int): Number of retries for OCR requests.
        ocr_jpeg_quality (int): JPEG quality setting for image compression before OCR.
        
    Returns:
        list: The combined results of all processing mappings for all files.
    """
    os.makedirs(output_dir, exist_ok=True)
    files_to_process = collect_input_files(inputs)

    if not files_to_process:
        print("[ERROR] No valid input files found.")
        return []

    print(f"Found {len(files_to_process)} file(s) to process.")
    combined = []
    for idx, file_path in enumerate(files_to_process, start=1):
        print(f"[{idx}/{len(files_to_process)}] Start: {file_path}")
        try:
            file_result = process_single_file(
                file_path=file_path,
                ocr_api_url=ocr_api_url,
                translation_api_url=translation_api_url,
                output_dir=output_dir,
                font_path=font_path,
                padding=padding,
                ocr_connect_timeout=ocr_connect_timeout,
                ocr_read_timeout=ocr_read_timeout,
                ocr_retries=ocr_retries,
                ocr_jpeg_quality=ocr_jpeg_quality,
            )
            combined.extend(file_result)
        except requests.exceptions.RequestException as e:
            print(f"[ERROR] API request failed for {file_path}: {e}")
        except (OSError, ValueError, IndexError, TimeoutError, RuntimeError) as e:
            print(f"[ERROR] Processing failed for {file_path}: {e}")

    combined_path = os.path.join(output_dir, "all_files_jp_th.json")
    with open(combined_path, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print(f"Saved combined json: {combined_path}")

    return combined


# Example usage:
INPUTS = ["images"]
result = process_multiple_files_with_new_apis(
    inputs=INPUTS,
    ocr_api_url=OCR_API_URL,
    translation_api_url=TRANSLATION_API_URL,
    font_path=FONT_PATH,
    padding=8,
    ocr_connect_timeout=20,
    ocr_read_timeout=300,
    ocr_retries=2,
    ocr_jpeg_quality=95,
)


[DEBUG][INPUT] Collecting files from 1 input item(s)
Found 5 file(s) to process.
[1/5] Start: images\0.jpg
[DEBUG][PIPELINE] Start processing: images\0.jpg
[DEBUG][IMAGE] Processing image file: images\0.jpg
[DEBUG][OCR] Uploading in-memory image: 0.jpg
[DEBUG][OCR] Attempt 1/3 -> https://your-ocr-endpoint.ngrok-free.app/predict (connect=20s, read=300s)
[DEBUG][ERROR] OCR request failed for 0.jpg: HTTPSConnectionPool(host='your-ocr-endpoint.ngrok-free.app', port=443): Max retries exceeded with url: /predict (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)')))
[DEBUG][OCR] Retrying after 1s
[DEBUG][OCR] Attempt 2/3 -> https://your-ocr-endpoint.ngrok-free.app/predict (connect=20s, read=300s)
[DEBUG][ERROR] OCR request failed for 0.jpg: HTTPSConnectionPool(host='your-ocr-endpoint.ngrok-free.app', port=443): Max retries exceeded with url: /predict (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)')))
[DEBUG][OCR] R